# 06 Spark Sql - Annotated Notebook

This notebook includes step-by-step markdown sections and English code comments.

## Step 1 - Import required libraries

In [1]:
# Import required libraries.
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

22/02/17 22:43:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


## Step 2 - Load data into a Spark DataFrame

In [2]:
# Load data into a Spark DataFrame.
df_green = spark.read.parquet('data/pq/green/*/*')

## Step 3 - Run this processing step

In [ ]:
# Run this processing step.


## Step 4 - Run the next processing step

In [16]:
# Run the next processing step.
df_green = df_green \
    .withColumnRenamed('lpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('lpep_dropoff_datetime', 'dropoff_datetime')

## Step 5 - Load data into a Spark DataFrame

In [5]:
# Load data into a Spark DataFrame.
df_yellow = spark.read.parquet('data/pq/yellow/*/*')

## Step 6 - Run the next processing step

In [19]:
# Run the next processing step.
df_yellow = df_yellow \
    .withColumnRenamed('tpep_pickup_datetime', 'pickup_datetime') \
    .withColumnRenamed('tpep_dropoff_datetime', 'dropoff_datetime')

## Step 7 - Run the next processing step

In [22]:
# Run the next processing step.
common_colums = []

yellow_columns = set(df_yellow.columns)

for col in df_green.columns:
    if col in yellow_columns:
        common_colums.append(col)

## Step 8 - Import required libraries

In [26]:
# Import required libraries.
from pyspark.sql import functions as F

## Step 9 - Run the next processing step

In [28]:
# Run the next processing step.
df_green_sel = df_green \
    .select(common_colums) \
    .withColumn('service_type', F.lit('green'))

## Step 10 - Run the next processing step

In [29]:
# Run the next processing step.
df_yellow_sel = df_yellow \
    .select(common_colums) \
    .withColumn('service_type', F.lit('yellow'))

## Step 11 - Combine DataFrames from multiple sources

In [30]:
# Combine DataFrames from multiple sources.
df_trips_data = df_green_sel.unionAll(df_yellow_sel)

## Step 12 - Inspect DataFrame content and size

In [33]:
# Inspect DataFrame content and size.
df_trips_data.groupBy('service_type').count().show()

+------------+--------+
|service_type|   count|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



## Step 13 - Run the next processing step

In [40]:
# Run the next processing step.
df_trips_data.columns

['VendorID',
 'pickup_datetime',
 'dropoff_datetime',
 'store_and_fwd_flag',
 'RatecodeID',
 'PULocationID',
 'DOLocationID',
 'passenger_count',
 'trip_distance',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'payment_type',
 'congestion_surcharge',
 'service_type']

## Step 14 - Register a temporary SQL view

In [35]:
# Register a temporary SQL view.
df_trips_data.registerTempTable('trips_data')

## Step 15 - Run a Spark SQL query

In [38]:
# Run a Spark SQL query.
spark.sql("""
SELECT
    service_type,
    count(1)
FROM
    trips_data
GROUP BY 
    service_type
""").show()

+------------+--------+
|service_type|count(1)|
+------------+--------+
|       green| 2304517|
|      yellow|39649199|
+------------+--------+



## Step 16 - Run a Spark SQL query

In [ ]:
# Run a Spark SQL query.
df_result = spark.sql("""
SELECT 
    -- Revenue grouping 
    PULocationID AS revenue_zone,
    date_trunc('month', pickup_datetime) AS revenue_month, 
    service_type, 

    -- Revenue calculation 
    SUM(fare_amount) AS revenue_monthly_fare,
    SUM(extra) AS revenue_monthly_extra,
    SUM(mta_tax) AS revenue_monthly_mta_tax,
    SUM(tip_amount) AS revenue_monthly_tip_amount,
    SUM(tolls_amount) AS revenue_monthly_tolls_amount,
    SUM(improvement_surcharge) AS revenue_monthly_improvement_surcharge,
    SUM(total_amount) AS revenue_monthly_total_amount,
    SUM(congestion_surcharge) AS revenue_monthly_congestion_surcharge,

    -- Additional calculations
    AVG(passenger_count) AS avg_monthly_passenger_count,
    AVG(trip_distance) AS avg_monthly_trip_distance
FROM
    trips_data
GROUP BY
    1, 2, 3
""")

## Step 17 - Write results to storage

In [49]:
# Write results to storage.
df_result.coalesce(1).write.parquet('data/report/revenue/', mode='overwrite')

## Step 18 - Run this processing step

In [ ]:
# Run this processing step.
